In [1]:
#cell1
# Setup imports

import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from google.colab import drive
from IPython.display import display

In [2]:
#cell2
# Mount Google Drive silently

import io
import contextlib

with contextlib.redirect_stdout(io.StringIO()):
    drive.mount("/content/drive", force_remount=False)

In [3]:
#cell3
# Define file paths, dataset names, and model names for logicRAG answers

BASE_DIR = Path("/content/drive/MyDrive/final_project/logicRAG/answer")

ANSWER_FILES = [
    {
        "dataset": "2wikimultihopqa",
        "model": "gemma4",
        "file_name": "2wikimultihopqa_gemma4_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_gemma4_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "gemma4",
        "file_name": "hotpotqa_gemma4_answers.json",
        "file_path": BASE_DIR / "hotpotqa_gemma4_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "gpt-oss-120b",
        "file_name": "2wikimultihopqa_gpt_oss_120b_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_gpt_oss_120b_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "gpt-oss-120b",
        "file_name": "hotpotqa_gpt_oss_120b_answers.json",
        "file_path": BASE_DIR / "hotpotqa_gpt_oss_120b_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "qwen3.5",
        "file_name": "2wikimultihopqa_qwen3.5_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_qwen3.5_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "qwen3.5",
        "file_name": "hotpotqa_qwen3.5_answers.json",
        "file_path": BASE_DIR / "hotpotqa_qwen3.5_answers.json",
    },
]

# Check that all required files exist before running the evaluation
missing_files = [
    str(file_info["file_path"])
    for file_info in ANSWER_FILES
    if not file_info["file_path"].exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following answer files were not found:\n" + "\n".join(missing_files)
    )

print(f"All {len(ANSWER_FILES)} answer files were found.")

All 6 answer files were found.


In [4]:
#cell4
# Normalize text and tokenize answers

def normalize_text(text):
    """
    Basic normalization for token-level comparison.
    Lowercase, remove punctuation, and normalize spaces.
    """
    if text is None:
        text = ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()

    chars = []
    for ch in text:
        if unicodedata.category(ch).startswith("P"):
            chars.append(" ")
        else:
            chars.append(ch)

    text = "".join(chars)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    """
    Convert answer text to a set of tokens.
    This keeps the same set-overlap logic as the previous notebook.
    """
    normalized = normalize_text(text)
    if not normalized:
        return set()
    return set(normalized.split())

In [5]:
#cell5
# Compute Precision, Recall, and F1 for one example

def compute_token_f1(predicted_answer, ground_truth_answer):
    """
    Compute token-level Precision, Recall, and F1.
    """
    pred_tokens = tokenize(predicted_answer)
    gt_tokens = tokenize(ground_truth_answer)

    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0, 0.0, 0.0

    overlap = pred_tokens.intersection(gt_tokens)

    precision = len(overlap) / len(pred_tokens)
    recall = len(overlap) / len(gt_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return precision, recall, f1

In [6]:
#cell6
# Load one JSON file and compute row-level scores

def load_json_file(file_path):
    """
    Load a JSON answer file.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected a list of records in: {file_path}")

    return data


def evaluate_file(dataset_name, model_name, file_name, file_path):
    """
    Evaluate all questions for one dataset-model file.
    """
    data = load_json_file(file_path)
    rows = []

    for idx, item in enumerate(data):
        gt = item.get("gt", "")
        response = item.get("response", "")
        question_type = item.get("type", "unknown")

        precision, recall, f1 = compute_token_f1(
            predicted_answer=response,
            ground_truth_answer=gt
        )

        rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "file_name": file_name,
            "row_index": idx,
            "source_index": item.get("source_index", idx),
            "type": question_type if question_type is not None else "unknown",
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "question": item.get("question", ""),
            "gt": gt,
            "response": response,
        })

    return pd.DataFrame(rows)

In [7]:
#cell7
# Evaluate all files while keeping dataset, model, and file identity separated

def evaluate_all_files(answer_files):
    """
    Evaluate every JSON file and combine row-level results.
    Each row keeps dataset, model, and file_name to prevent mixing results.
    """
    all_dfs = []

    for file_info in answer_files:
        file_df = evaluate_file(
            dataset_name=file_info["dataset"],
            model_name=file_info["model"],
            file_name=file_info["file_name"],
            file_path=file_info["file_path"],
        )
        all_dfs.append(file_df)

    return pd.concat(all_dfs, ignore_index=True)

In [8]:
#cell8
# Build overall and per-type summaries for each separate file

def build_summary(scores_df):
    """
    Create overall and per-type macro summaries.
    Results are grouped by dataset, model, and file_name so files do not get mixed.
    """
    group_cols = ["dataset", "model", "file_name"]

    overall_df = (
        scores_df
        .groupby(group_cols, as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )
    overall_df["type"] = "overall"

    type_df = (
        scores_df
        .groupby(group_cols + ["type"], as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )

    summary_df = pd.concat([overall_df, type_df], ignore_index=True)

    # Convert macro F1 to percentage for easier reporting
    summary_df["f1_percent"] = summary_df["f1_macro"] * 100

    # Keep the display order consistent with the new logicRAG files
    dataset_order = ["2wikimultihopqa", "hotpotqa"]
    model_order = ["gemma4", "gpt-oss-120b", "qwen3.5"]

    summary_df["dataset"] = pd.Categorical(
        summary_df["dataset"],
        categories=dataset_order,
        ordered=True
    )

    summary_df["model"] = pd.Categorical(
        summary_df["model"],
        categories=model_order,
        ordered=True
    )

    # Put the overall row before the question-type rows
    summary_df["type_sort"] = summary_df["type"].apply(
        lambda x: "000_overall" if x == "overall" else str(x)
    )

    summary_df = (
        summary_df
        .sort_values(["model", "dataset", "file_name", "type_sort"])
        .drop(columns=["type_sort"])
        .reset_index(drop=True)
    )

    column_order = [
        "dataset",
        "model",
        "file_name",
        "type",
        "n_questions",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "f1_percent",
    ]
    summary_df = summary_df[column_order]

    numeric_cols = ["precision_macro", "recall_macro", "f1_macro", "f1_percent"]
    summary_df[numeric_cols] = summary_df[numeric_cols].round(6)

    return summary_df

In [9]:
#cell9
# Final output: separate detailed summary for every dataset-model file

scores_df = evaluate_all_files(ANSWER_FILES)
summary_df = build_summary(scores_df)

for file_name in summary_df["file_name"].unique():
    print("=" * 100)
    print(f"Results for file: {file_name}")
    print("=" * 100)

    file_summary = summary_df[summary_df["file_name"] == file_name].reset_index(drop=True)
    display(file_summary)

Results for file: 2wikimultihopqa_gemma4_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,overall,1000,0.694573,0.729079,0.700325,70.032540
1,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,bridge_comparison,250,0.601333,0.600500,0.600727,60.072727
2,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,comparison,250,0.928000,0.928000,0.928000,92.800000
3,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,compositional,250,0.507565,0.606200,0.531725,53.172527
4,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,inference,250,0.741395,0.781614,0.740849,74.084906


Results for file: hotpotqa_gemma4_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,hotpotqa,gemma4,hotpotqa_gemma4_answers.json,overall,1000,0.667034,0.652600,0.642988,64.298805
1,hotpotqa,gemma4,hotpotqa_gemma4_answers.json,bridge,700,0.621953,0.610299,0.597612,59.761218
2,hotpotqa,gemma4,hotpotqa_gemma4_answers.json,comparison,300,0.772222,0.751301,0.748865,74.886508


Results for file: 2wikimultihopqa_gpt_oss_120b_answers.json


...

,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,hotpotqa,gpt-oss-120b,hotpotqa_gpt_oss_120b_answers.json,overall,1000,0.698189,0.673103,0.668373,66.837301
1,hotpotqa,gpt-oss-120b,hotpotqa_gpt_oss_120b_answers.json,bridge,700,0.652547,0.626232,0.621108,62.110770
2,hotpotqa,gpt-oss-120b,hotpotqa_gpt_oss_120b_answers.json,comparison,300,0.804688,0.782468,0.778659,77.865873


Results for file: 2wikimultihopqa_qwen3.5_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,overall,1000,0.688594,0.729013,0.697187,69.718663
1,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,bridge_comparison,250,0.572197,0.578405,0.574428,57.442817
2,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,comparison,250,0.928571,0.929000,0.928727,92.872727
3,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,compositional,250,0.520727,0.618867,0.543805,54.380526
4,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,inference,250,0.732881,0.789781,0.741786,74.178581


Results for file: hotpotqa_qwen3.5_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,hotpotqa,qwen3.5,hotpotqa_qwen3.5_answers.json,overall,1000,0.676764,0.654487,0.649729,64.972949
1,hotpotqa,qwen3.5,hotpotqa_qwen3.5_answers.json,bridge,700,0.623758,0.600733,0.595132,59.513170
2,hotpotqa,qwen3.5,hotpotqa_qwen3.5_answers.json,comparison,300,0.800444,0.779912,0.777124,77.712434


In [10]:
#cell10
# Optional comparison table for easier model comparison by dataset and type

comparison_df = (
    summary_df
    .pivot_table(
        index=["dataset", "type"],
        columns="model",
        values="f1_percent",
        aggfunc="first"
    )
    .reset_index()
)

comparison_df.columns.name = None

display(comparison_df)

/tmp/ipykernel_17005/3631904528.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


,dataset,type,gemma4,gpt-oss-120b,qwen3.5
0,2wikimultihopqa,bridge_comparison,60.072727,69.000000,57.442817
1,2wikimultihopqa,comparison,92.800000,92.800000,92.872727
2,2wikimultihopqa,compositional,53.172527,57.884043,54.380526
3,2wikimultihopqa,inference,74.084906,77.098192,74.178581
4,2wikimultihopqa,overall,70.032540,74.195559,69.718663
5,hotpotqa,bridge,59.761218,62.110770,59.513170
6,hotpotqa,comparison,74.886508,77.865873,77.712434
7,hotpotqa,overall,64.298805,66.837301,64.972949
